# Notebook 01 — Mod30 Residue Manifold

**Repo:** `mod30-manifold-tiling`  
**Purpose:** build a clean arithmetic analogue for the manifold-tiling behavior described in arXiv:2604.28119v1.

The paper’s useful interpretability idea is that sparse feature systems may recover curved concept structure through **local tiling** rather than one clean global feature direction.  
This notebook gives a finite arithmetic version:

```text
integers → mod30 residue manifold
constraint gate → 8 surviving lanes
observable signal → local residue tiles
```

Mod30 prime-candidate lanes:

```text
1, 7, 11, 13, 17, 19, 23, 29
```


## 0. Setup

This cell works both locally and in Colab-style notebook execution.  
It adds the repo root to `sys.path`, so notebook cells can import from `src/`.


In [ ]:
from pathlib import Path
import sys

# Find repo root when running from notebooks/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"

for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Figures:", FIGURES_DIR)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.mod30 import (
    MOD30,
    MOD30_RESIDUES,
    mod30_index,
    mod30_mask,
    mod30_residues,
    generate_coprime_residues,
)
from src.tiling_metrics import lane_density, gate_summary
from src.plots import save_current

MOD30_RESIDUES


## 1. Full mod30 residue manifold

Start with integers and map each integer to a residue class modulo 30.

This is the finite ambient structure: 30 possible residue states.


In [ ]:
n_min = 1
n_max = 300
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})
df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))

df.head(12)


In [ ]:
plt.figure(figsize=(12, 4.5))
plt.scatter(df["n"], df["mod30_residue"], s=14)
plt.yticks(range(0, 30, 2))
plt.xlabel("integer n")
plt.ylabel("n mod 30")
plt.title("Full mod30 residue manifold: all 30 residue states")
save_current(FIGURES_DIR / "01_full_mod30_residue_manifold.png")
plt.show()


## 2. Apply the finite constraint gate

The mod30 gate keeps the residue classes coprime to 30:

```text
1, 7, 11, 13, 17, 19, 23, 29
```

This is not primality. It is the first coarse wheel filter after excluding divisibility by 2, 3, and 5.


In [ ]:
summary = gate_summary(values.tolist(), mod30_mask)
summary


In [ ]:
inside = df[df["inside_mod30_gate"]]
outside = df[~df["inside_mod30_gate"]]

plt.figure(figsize=(12, 4.8))
plt.scatter(outside["n"], outside["mod30_residue"], s=10, alpha=0.35, label="outside gate")
plt.scatter(inside["n"], inside["mod30_residue"], s=22, label="inside mod30 gate")
plt.yticks(range(30))
plt.xlabel("integer n")
plt.ylabel("n mod 30")
plt.title("Mod30 constraint gate: 8 local residue lanes persist")
plt.legend()
save_current(FIGURES_DIR / "02_mod30_gate_highlighted_lanes.png")
plt.show()


## 3. Lane density

The gate does not collapse the structure into one direction.  
It distributes candidate signal across 8 local lanes.

That is the arithmetic analogue of a tiled representation:

```text
global structure exists
observable signal appears as local tiles
```


In [ ]:
density = lane_density(inside["n"].tolist(), mod30_index)
density_df = pd.DataFrame({
    "residue": list(density.keys()),
    "density_inside_gate": list(density.values()),
})
density_df


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(density_df["residue"], density_df["density_inside_gate"])
plt.xticks(MOD30_RESIDUES)
plt.xlabel("mod30 residue lane")
plt.ylabel("density among gated candidates")
plt.title("Eight mod30 lanes tile the surviving candidate structure")
save_current(FIGURES_DIR / "03_mod30_lane_density.png")
plt.show()


## 4. Residue histogram: all residues vs gated lanes

This plot gives the paper-facing punchline:

- all residues form the finite ambient manifold
- the gate selects structured local tiles
- the selected tiles repeat periodically


In [ ]:
all_counts = df["mod30_residue"].value_counts().sort_index()
gate_counts = inside["mod30_residue"].value_counts().sort_index()

hist_df = pd.DataFrame({
    "residue": range(30),
    "all_count": [all_counts.get(r, 0) for r in range(30)],
    "gate_count": [gate_counts.get(r, 0) for r in range(30)],
    "inside_gate": [r in MOD30_RESIDUES for r in range(30)],
})

hist_df.to_csv(DATA_DIR / "01_mod30_residue_histogram.csv", index=False)
hist_df.head()


In [ ]:
plt.figure(figsize=(12, 4.8))
plt.bar(hist_df["residue"], hist_df["all_count"], alpha=0.35, label="all residues")
plt.bar(hist_df["residue"], hist_df["gate_count"], label="inside mod30 gate")
plt.xticks(range(30))
plt.xlabel("residue class mod 30")
plt.ylabel("count")
plt.title("Local residue tiles inside a finite mod30 manifold")
plt.legend()
save_current(FIGURES_DIR / "04_all_vs_gated_residue_histogram.png")
plt.show()


## 5. Map to arXiv:2604.28119v1

This notebook uses Mod30 as a finite toy analogue, not as a claim about neural networks.

| arXiv paper concept | Mod30 analogue |
|---|---|
| concept manifold | finite residue manifold |
| SAE local tiling | surviving residue lanes |
| feature dilution | structure distributed over multiple lanes |
| global direction failure | no single residue lane captures the whole structure |
| interpretable local detectors | lane-wise residue filters |

The practical claim:

```text
A clean structure can be globally simple while locally tiled.
```


## 6. Foreshadowing: mod210 and mod2310

Mod30 is the first useful compact example.  
Higher primorial wheels refine the tiling:

```text
mod30   = 2 × 3 × 5        → 8 lanes
mod210  = 2 × 3 × 5 × 7    → 48 lanes
mod2310 = 2 × 3 × 5 × 7 × 11 → 480 lanes
```

The same pipeline scales by replacing the modulus and coprime residue set.


In [ ]:
# Foreshadow only: no heavy analysis yet.

mod30_lanes = generate_coprime_residues(30)
mod210_lanes = generate_coprime_residues(210)
mod2310_lanes = generate_coprime_residues(2310)

pd.DataFrame({
    "modulus": [30, 210, 2310],
    "lane_count": [len(mod30_lanes), len(mod210_lanes), len(mod2310_lanes)],
    "interpretation": [
        "coarse finite tiling",
        "finer primorial tiling",
        "next refined primorial tiling",
    ],
})


In [ ]:
primorial_df = pd.DataFrame({
    "modulus": [30, 210, 2310],
    "lane_count": [len(mod30_lanes), len(mod210_lanes), len(mod2310_lanes)],
})
primorial_df.to_csv(DATA_DIR / "01_primorial_lane_counts.csv", index=False)

plt.figure(figsize=(7, 4.5))
plt.plot(primorial_df["modulus"].astype(str), primorial_df["lane_count"], marker="o")
plt.xlabel("primorial modulus")
plt.ylabel("coprime residue lanes")
plt.title("Foreshadowing refinement: mod30 → mod210 → mod2310")
save_current(FIGURES_DIR / "05_primorial_lane_count_refinement.png")
plt.show()


## 7. Save compact notebook summary

This creates a small Markdown summary beside the CSV + figures.


In [ ]:
summary_md = f"""# Notebook 01 Summary — Mod30 Residue Manifold

This notebook builds a finite arithmetic analogue for local manifold tiling.

## Core result

- Mod30 has 30 residue states.
- Excluding divisibility by 2, 3, and 5 leaves 8 residue lanes:
  `{MOD30_RESIDUES}`
- These lanes tile candidate structure locally rather than collapsing it into one global direction.

## arXiv:2604.28119v1 bridge

Sparse feature systems may tile concept manifolds locally.
Mod30 gives a transparent finite analogue:

- finite manifold: residues modulo 30
- local tiles: coprime residue lanes
- dilution: structure distributed across multiple lanes

## Generated files

- `figures/01_full_mod30_residue_manifold.png`
- `figures/02_mod30_gate_highlighted_lanes.png`
- `figures/03_mod30_lane_density.png`
- `figures/04_all_vs_gated_residue_histogram.png`
- `figures/05_primorial_lane_count_refinement.png`
- `data/01_mod30_residue_histogram.csv`
- `data/01_primorial_lane_counts.csv`
"""

summary_path = OUTPUTS_DIR / "01_mod30_residue_manifold_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 8. Optional: zip-download pattern for Colab

Uncomment the next cell when running in Colab or when you want a single downloadable bundle containing figures, data, and Markdown outputs.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_01_mod30_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 9. Next notebook direction

Recommended Notebook 02:

```text
02_local_tiling_vs_global_capture.ipynb
```

Goal:

- compare one global mod30 signal against 8 local lane detectors
- show why local tiled capture can preserve structure better than a single global summary
- connect more explicitly to SAE feature fragmentation / dilution
```
